# Unit 3: 数据采集深入

## 学习目标
- 了解不同 OpenBCI 板卡的连接配置
- 掌握流式保存数据到 CSV 文件
- 使用 Playback File Board 回放数据
- 使用 Marker 标记事件
- 使用 DataFilter 读写文件

In [14]:
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
# 负责采集数据
from brainflow.board_shim import (
    BoardShim, 
    BrainFlowInputParams, 
    BoardIds, 
    BrainFlowPresets
)
# 负责处理数据
from brainflow.data_filter import (
    DataFilter
)
# 负责机器学习
from brainflow.ml_model import (
    MLModel
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("模块导入完成")

模块导入完成


## 3.1 不同板卡的连接配置

BrainFlow 的最大优势：**切换硬件只需修改 `board_id` 和 `BrainFlowInputParams`**。

### 配置速查表

| 板卡 | board_id | 需设置参数 |
|------|----------|------------|
| Synthetic Board | `BoardIds.SYNTHETIC_BOARD` | 无需任何参数 |
| Cyton (USB) | `BoardIds.CYTON_BOARD` | `serial_port` (e.g. "COM3") |
| Cyton+Daisy | `BoardIds.CYTON_DAISY_BOARD` | `serial_port` (e.g. "COM3") |
| Ganglion (蓝牙) | `BoardIds.GANGLION_BOARD` | `serial_port` 或 `mac_address` |
| Cyton WiFi | `BoardIds.CYTON_WIFI_BOARD` | `ip_address`, `ip_port` |
| Playback File | `BoardIds.PLAYBACK_FILE_BOARD` | `file`, `master_board` |

### 代码示例（仅展示配置差异，不会真正连接硬件）

In [3]:
def show_board_config(name, board_id, params_override=None):
    """展示板卡配置示例"""
    params = BrainFlowInputParams()
    if params_override:
        for k, v in params_override.items():
            setattr(params, k, v)
    descr = BoardShim.get_board_descr(board_id)
    print(f"\n{'='*50}")
    print(f"板卡: {name}")
    print(f"Board ID: {board_id}")
    print(f"采样率: {descr.get('sampling_rate')} Hz")
    print(f"EEG 通道数: {len(descr.get('eeg_channels', []))}")
    print(f"EEG 通道名: {descr.get('eeg_names', 'N/A')}")
    if params_override:
        print(f"连接参数: {params_override}")

# 展示各板卡配置（仅信息查询，不连接硬件）
show_board_config("Synthetic Board", BoardIds.SYNTHETIC_BOARD)
show_board_config("Cyton 8CH", BoardIds.CYTON_BOARD, {'serial_port': 'COM3'})
show_board_config("Cyton + Daisy 16CH", BoardIds.CYTON_DAISY_BOARD, {'serial_port': 'COM3'})
show_board_config("Ganglion 4CH", BoardIds.GANGLION_BOARD, {'serial_port': 'COM5'})
show_board_config(
    "Cyton WiFi", 
    BoardIds.CYTON_WIFI_BOARD, 
    params_override={
    'ip_address': '192.168.4.1', 
    'ip_port': 6677
    }
)


板卡: Synthetic Board
Board ID: -1
采样率: 250 Hz
EEG 通道数: 16
EEG 通道名: Fz,C3,Cz,C4,Pz,PO7,Oz,PO8,F5,F7,F3,F1,F2,F4,F6,F8

板卡: Cyton 8CH
Board ID: 0
采样率: 250 Hz
EEG 通道数: 8
EEG 通道名: Fp1,Fp2,C3,C4,P7,P8,O1,O2
连接参数: {'serial_port': 'COM3'}

板卡: Cyton + Daisy 16CH
Board ID: 2
采样率: 125 Hz
EEG 通道数: 16
EEG 通道名: Fp1,Fp2,C3,C4,P7,P8,O1,O2,F7,F8,F3,F4,T7,T8,P3,P4
连接参数: {'serial_port': 'COM3'}

板卡: Ganglion 4CH
Board ID: 1
采样率: 200 Hz
EEG 通道数: 4
EEG 通道名: N/A
连接参数: {'serial_port': 'COM5'}

板卡: Cyton WiFi
Board ID: 5
采样率: 1000 Hz
EEG 通道数: 8
EEG 通道名: N/A
连接参数: {'ip_address': '192.168.4.1', 'ip_port': 6677}


## 3.2 流式保存数据 —— add_streamer

BrainFlow 支持在采集数据的同时**直接写入文件**，无需先获取再保存。
这对于长时间记录非常有用。

In [8]:
from brainflow.board_shim import BoardShim, BoardIds, BrainFlowPresets
board_id = BoardIds.SYNTHETIC_BOARD.value
# 获取设备的所有预设
presets = BoardShim.get_board_presets(board_id)
for p in presets:
    preset = BrainFlowPresets(p)
    print(p, preset.name)

1 AUXILIARY_PRESET
0 DEFAULT_PRESET


In [17]:
# 使用 add_streamer 将数据流直接写入 CSV 文件
output_file = "../data/synthetic_recording.csv"
os.makedirs("../data", exist_ok=True)

# 关闭日志记录
BoardShim.disable_board_logger()

board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
print("添加流式写入器")
# add_streamer 在 start_stream 之前调用
# 格式: "file://路径:w" (w=覆盖写入, a=追加写入)
board.add_streamer(f"file://{output_file}:w", BrainFlowPresets.DEFAULT_PRESET)
print(f"数据流将保存到: {output_file}")

board.start_stream()
time.sleep(5)

# 获取内存中的数据
data = board.get_board_data()
print(f"内存数据形状: {data.shape}")

board.stop_stream()
board.release_session()

# 验证文件
file_size = os.path.getsize(output_file)
print(f"\nCSV 文件大小: {file_size:,} 字节")


添加流式写入器
数据流将保存到: ../data/synthetic_recording.csv
内存数据形状: (32, 1251)

CSV 文件大小: 421,326 字节


In [28]:

# 读取并加载 CSV 文件
data = DataFilter.read_file(output_file)
print(f"CSV 文件形状: {data.shape}")
# 在 pandas 中，时间是行索引，通道是列索引
# 因此，需要将数据转置，使时间成为行索引，通道成为列索引
df = pd.DataFrame(data.T)
print(f"DataFrame 形状: {df.shape}")
df.head()


CSV 文件形状: (32, 1252)
DataFrame 形状: (1252, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.0,11.809352,28.917265,47.572211,71.598351,103.266636,133.329572,317.356853,420.188083,484.913265,...,0.817442,0.928813,523.734583,259308.024084,36.609399,972.258369,1079.059095,94.434984,1.778863e+09,0.0
1,1.0,13.661872,33.430519,62.634581,99.563616,110.025576,129.532795,319.852349,365.809973,336.441176,...,0.879202,0.905171,471.619907,236660.881571,36.606630,1076.082005,978.030562,97.919248,1.778863e+09,0.0
2,2.0,15.092080,40.786994,66.899142,104.992515,111.230516,108.679156,91.067497,51.215281,25.306298,...,0.861426,1.062087,487.603861,229994.365841,36.606385,981.816163,990.210653,80.190564,1.778863e+09,0.0
3,3.0,17.085686,46.042725,68.900047,94.939894,75.078139,52.249482,16.468465,-24.991343,-33.530657,...,0.889186,1.006180,547.732971,261397.258301,36.609195,933.289588,908.625031,84.614778,1.778863e+09,0.0
4,4.0,18.126755,44.596335,64.641444,64.939578,34.027253,-7.934742,-0.094569,-2.676587,57.581721,...,0.850331,0.966522,461.740208,274543.231403,36.595742,1098.142578,1037.659542,97.715742,1.778863e+09,0.0


## 3.3 DataFilter 读写文件

BrainFlow 提供了高效的二进制文件读写（比 pandas.to_csv 更快）：

In [29]:
# 使用 DataFilter 写入和读取数据文件
bf_file = "../data/brainflow_data.csv"

# 采集数据
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(3)
data = board.get_board_data()
board.stop_stream()
board.release_session()

print(f"原始数据形状: {data.shape}")

# 写入文件（BrainFlow 格式）
DataFilter.write_file(data, bf_file, 'w')  # 'w' = 覆盖, 'a' = 追加
print(f"数据已写入: {bf_file}")

# 从文件读取
restored_data = DataFilter.read_file(bf_file)
print(f"读取数据形状: {restored_data.shape}")
# 验证数据是否一致
# abs(a - b) <= atol + rtol * abs(b)
# atol: 绝对容差
# rtol: 相对容差
print(f"数据一致: {np.allclose(data, restored_data, atol=1e-6, rtol=1e-6)}")


原始数据形状: (32, 751)
数据已写入: ../data/brainflow_data.csv
读取数据形状: (32, 751)
数据一致: True


## 3.4 Playback File Board —— 离线回放

Playback File Board 可以回放之前录制的数据文件，这对于离线分析和算法调试非常有价值。

**关键参数**：
- `file`: 回放文件的路径
- `master_board`: 原始采集板卡的类型（告诉 BrainFlow 如何解析数据格式）

In [63]:
# 使用 Playback File Board 回放刚才录制的数据
params = BrainFlowInputParams()
params.file = bf_file
params.master_board = BoardIds.SYNTHETIC_BOARD  # 关键：指定原始板卡类型

playback_board = BoardShim(BoardIds.PLAYBACK_FILE_BOARD, params)
playback_board.prepare_session()

playback_board.config_board("old_timestamps")  # 使用原始时间戳

playback_board.start_stream()

# 回放时数据按时间顺序到达，等待足够数据
print("回放进行中...")
max_wait = 10  # 最多等待10秒
elapsed = 0
# 只要回放缓冲区里的样本数还小于原始样本数，并且等待时间还没超过 10 秒，就继续等。
while playback_board.get_board_data_count() < data.shape[1] and elapsed < max_wait:
    time.sleep(0.2)
    elapsed += 0.2

playback_data = playback_board.get_board_data()
playback_board.stop_stream()
playback_board.release_session()

print(f"原始数据形状:    {data.shape}")
print(f"回放数据形状:    {playback_data.shape}")
print(f"数据接近一致:    {np.allclose(data[:, :playback_data.shape[1]], playback_data, rtol=1e-6, atol=1e-6,)}")




回放进行中...
原始数据形状:    (32, 751)
回放数据形状:    (32, 751)
数据接近一致:    True


### Playback Board 高级配置

可以通过 `config_board` 控制回放行为：

In [67]:
# 回放配置示例
params = BrainFlowInputParams()
params.file = bf_file
params.master_board = BoardIds.SYNTHETIC_BOARD

board = BoardShim(BoardIds.PLAYBACK_FILE_BOARD, params)
board.prepare_session()

# 配置回放行为（在 start_stream 之前设置）
board.config_board("loopback_true")   # 循环回放
# board.config_board("loopback_false")  # 不循环（默认）
# board.config_board("new_timestamps")  # 生成新时间戳（默认）
# board.config_board("old_timestamps")  # 使用原始时间戳

board.start_stream()
time.sleep(5)
playback_data = board.get_board_data()
board.stop_stream()
board.release_session()

print(f"回放数据量: {playback_data.shape[1]} 样本")

回放数据量: 1245 样本


## 3.5 使用 Marker 标记事件

Marker 用于在数据流中标记特定事件（如刺激呈现、被试反应等），是实现事件相关分析的基础。

In [68]:
# 演示 Marker 的使用
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(1) # 先等待 1 秒，确保数据采集稳定


# 在不同时间点插入事件标记
events = [
    ("实验开始", 1),
    ("刺激 A 呈现", 2),
    ("刺激 B 呈现", 3),
    ("被试反应", 4),
    ("实验结束", 99),
]

for event_name, marker_value in events:
    board.insert_marker(marker_value)
    print(f"📌 Marker {marker_value}: {event_name}")
    time.sleep(0.5) # 每隔 0.5 秒插入一个标记，模拟实验中的时间间隔

time.sleep(1)
data = board.get_board_data()
board.stop_stream()
board.release_session()

# 分析 Marker 通道
descr = BoardShim.get_board_descr(BoardIds.SYNTHETIC_BOARD)
marker_ch = descr['marker_channel']
ts_ch = descr['timestamp_channel']

marker_data = data[marker_ch, :]

# 查找非零标记
marker_indices = np.where(marker_data != 0)[0]
print(f"\n找到 {len(marker_indices)} 个标记:")
for idx in marker_indices:
    ts = data[ts_ch, idx]
    print(f"  时间={ts:.0f}, Marker={marker_data[idx]:.0f}")

📌 Marker 1: 实验开始
📌 Marker 2: 刺激 A 呈现
📌 Marker 3: 刺激 B 呈现
📌 Marker 4: 被试反应
📌 Marker 99: 实验结束

找到 5 个标记:
  时间=1778871105, Marker=1
  时间=1778871105, Marker=2
  时间=1778871106, Marker=3
  时间=1778871106, Marker=4
  时间=1778871107, Marker=99


## 3.6 不同板卡的采样率对比

不同板卡有不同的采样率，这对信号处理有直接影响（如 Nyquist 频率）。

In [ ]:
# 各 OpenBCI 板卡的采样率对比
openbci_boards = [
    ("Synthetic", BoardIds.SYNTHETIC_BOARD),
    ("Cyton 8CH", BoardIds.CYTON_BOARD),
    ("Cyton+Daisy 16CH", BoardIds.CYTON_DAISY_BOARD),
    ("Ganglion 4CH", BoardIds.GANGLION_BOARD),
]

print(f"{'板卡':<20} {'采样率':<10} {'Nyquist':<10} {'通道数':<8} {'EEG 带宽'}")
print("-" * 70)
for name, bid in openbci_boards:
    d = BoardShim.get_board_descr(bid)
    sr = d['sampling_rate']
    n_ch = len(d.get('eeg_channels', []))
    nyquist = sr / 2
    print(f"{name:<20} {sr:<5} Hz   {nyquist:<5} Hz   {n_ch:<8} 0 ~ {nyquist} Hz")

## 3.7 综合练习：完整的数据记录与回放流程

模拟一个认知实验：
1. 采集 15 秒数据（含事件标记）
2. 保存为 BrainFlow 格式文件
3. 使用回放板加载并验证

In [ ]:
print("=== 模拟认知实验 ===\n")

# Step 1: 采集数据
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()

# 实验流程
time.sleep(2)        # 基线
board.insert_marker(10)  # 任务开始
print("📌 任务开始")
time.sleep(5)        # 任务执行
board.insert_marker(20)  # 任务结束
print("📌 任务结束")
time.sleep(3)        # 恢复期
board.insert_marker(99)  # 实验结束
print("📌 实验结束")

data = board.get_board_data()
board.stop_stream()
board.release_session()

# Step 2: 保存数据
exp_file = "data/experiment_demo.csv"
DataFilter.write_file(data, exp_file, 'w')
print(f"\n✅ 数据已保存到: {exp_file}")

# Step 3: 回放验证
params = BrainFlowInputParams()
params.file = exp_file
params.master_board = BoardIds.SYNTHETIC_BOARD

pb_board = BoardShim(BoardIds.PLAYBACK_FILE_BOARD, params)
pb_board.prepare_session()
pb_board.start_stream()

elapsed = 0
while pb_board.get_board_data_count() < data.shape[1] and elapsed < 15:
    time.sleep(0.2)
    elapsed += 0.2

playback_data = pb_board.get_board_data()
pb_board.stop_stream()
pb_board.release_session()

print(f"回放验证通过: 原始 {data.shape[1]} 样本, 回放 {playback_data.shape[1]} 样本")

# 可视化：标记实验阶段
fig, ax = plt.subplots(figsize=(14, 4))
eeg_ch = BoardShim.get_eeg_channels(BoardIds.SYNTHETIC_BOARD)
ax.plot(data[eeg_ch[0], :], alpha=0.7, label='Fz')

marker_ch = BoardShim.get_marker_channel(BoardIds.SYNTHETIC_BOARD)
for idx in np.where(data[marker_ch, :] != 0)[0]:
    mv = data[marker_ch, idx]
    ax.axvline(x=idx, color='red', linestyle='--', alpha=0.7, 
               label=f'Marker {mv:.0f}' if f'Marker {mv:.0f}' not in [l.get_label() for l in ax.lines] else '')

ax.legend()
ax.set_title('模拟认知实验 — Fz 通道 EEG 信号')
ax.set_xlabel('采样点')
ax.set_ylabel('μV')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.8 转换为 Pandas DataFrame 的最佳实践

在数据分析时，经常需要将 BrainFlow 数据转换为 Pandas DataFrame。

In [ ]:
def brainflow_to_dataframe(data, board_id):
    """将 BrainFlow 数据数组转换为带标签的 Pandas DataFrame"""
    descr = BoardShim.get_board_descr(board_id)
    eeg_channels = descr['eeg_channels']
    eeg_names = BoardShim.get_eeg_names(board_id)
    
    # 转置: (num_rows, num_samples) → (num_samples, num_rows)
    df = pd.DataFrame(np.transpose(data))
    
    # 构建列名
    col_names = [f'Row_{i}' for i in range(data.shape[0])]
    for ch_idx, name in zip(eeg_channels, eeg_names):
        col_names[ch_idx] = f'EEG_{name}'
    col_names[descr['timestamp_channel']] = 'Timestamp'
    col_names[descr['marker_channel']] = 'Marker'
    df.columns = col_names
    
    return df

# 测试
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(2)
data = board.get_board_data()
board.stop_stream()
board.release_session()

df = brainflow_to_dataframe(data, BoardIds.SYNTHETIC_BOARD)
print(f"DataFrame 形状: {df.shape}")
display(df.head(10))

## 小结

| 知识点 | 要点 |
|--------|------|
| 板卡配置 | 不同板卡只需改 `board_id` + `params`，代码不变 |
| `add_streamer` | 采集时直接写文件，适合长时间记录 |
| `DataFilter.write_file / read_file` | 高效的 BrainFlow 格式读写 |
| Playback Board | 用 `file` + `master_board` 回放历史数据 |
| Marker | `insert_marker()` 在数据流中标记事件 |

→ [Unit 4: 信号处理 API](unit4_signal_processing.ipynb) — 滤波、去噪、信号增强